In [1]:
# ==========================================================
# Globe Contours Dataset
# NLP + K-Means Clustering Analysis
# ==========================================================


# ----------------------------------------------------------
# 1. Import Libraries
# ----------------------------------------------------------

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import re
import nltk

from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from sklearn.decomposition import PCA

from scipy.sparse import hstack


# Download NLP stopwords
nltk.download("stopwords")


print("Libraries loaded successfully")


# ----------------------------------------------------------
# 2. Load Dataset
# ----------------------------------------------------------

# Make sure globe_contours.csv is in the same folder as notebook

df = pd.read_csv(
    "globe_contours.csv"
)


print("Dataset Loaded")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


# Display data

df.head()


# ----------------------------------------------------------
# 3. Dataset Information
# ----------------------------------------------------------

print(df.info())


print("\nMissing Values:")
print(df.isnull().sum())


# ----------------------------------------------------------
# 4. Data Cleaning
# ----------------------------------------------------------

# Remove duplicates

df = df.drop_duplicates()


# Replace missing values

df = df.fillna("Unknown")


print(
    "After Cleaning:",
    df.shape
)


# ----------------------------------------------------------
# 5. Detect Text Columns
# ----------------------------------------------------------

text_columns = df.select_dtypes(
    include=["object"]
).columns


print("Text Columns:")
print(list(text_columns))


# ----------------------------------------------------------
# 6. Combine Text Columns
# ----------------------------------------------------------

df["combined_text"] = df[
    text_columns
].astype(str).apply(
    lambda x: " ".join(x),
    axis=1
)


df[
    ["combined_text"]
].head()


# ----------------------------------------------------------
# 7. NLP Text Cleaning
# ----------------------------------------------------------

stop_words = set(
    stopwords.words("english")
)


def clean_text(text):

    # lowercase
    text = text.lower()

    # remove symbols
    text = re.sub(
        r"[^a-z\s]",
        "",
        text
    )

    # split words
    words = text.split()

    # remove stop words
    words = [
        word
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)



df["clean_text"] = df[
    "combined_text"
].apply(
    clean_text
)


df[
    ["clean_text"]
].head()


# ----------------------------------------------------------
# 8. TF-IDF Feature Extraction
# ----------------------------------------------------------

tfidf = TfidfVectorizer(
    max_features=500
)


text_features = tfidf.fit_transform(
    df["clean_text"]
)


print(
    "Text Feature Shape:",
    text_features.shape
)



# ----------------------------------------------------------
# 9. Extract Numeric Features
# ----------------------------------------------------------

numeric_features = df.select_dtypes(
    include=np.number
)


print(
    "Numeric Columns:"
)

print(
    numeric_features.columns
)



# ----------------------------------------------------------
# 10. Scale Numeric Data
# ----------------------------------------------------------

if numeric_features.shape[1] > 0:

    scaler = StandardScaler()

    numeric_scaled = scaler.fit_transform(
        numeric_features
    )

else:

    numeric_scaled = np.empty(
        (len(df),0)
    )



print(
    "Numeric Shape:",
    numeric_scaled.shape
)



# ----------------------------------------------------------
# 11. Combine NLP + Numeric Features
# ----------------------------------------------------------

X = hstack(
    [
        text_features,
        numeric_scaled
    ]
)


print(
    "Final Feature Matrix:",
    X.shape
)



# ----------------------------------------------------------
# 12. Find Best K Value
# ----------------------------------------------------------

silhouette_scores = []


k_values = range(
    2,
    11
)


for k in k_values:


    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )


    labels = kmeans.fit_predict(
        X
    )


    score = silhouette_score(
        X,
        labels
    )


    silhouette_scores.append(
        score
    )



plt.figure(
    figsize=(8,5)
)


plt.plot(
    list(k_values),
    silhouette_scores,
    marker="o"
)


plt.xlabel(
    "Number of Clusters (K)"
)


plt.ylabel(
    "Silhouette Score"
)


plt.title(
    "Optimal K Selection"
)


plt.grid()

plt.show()



# ----------------------------------------------------------
# 13. Apply Final K-Means Model
# ----------------------------------------------------------

# Change this value based on graph

optimal_k = 5


kmeans = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=10
)


df["cluster"] = kmeans.fit_predict(
    X
)


print(
    "Clustering Completed"
)



# ----------------------------------------------------------
# 14. Cluster Distribution
# ----------------------------------------------------------

print(
    df["cluster"].value_counts()
)



plt.figure(
    figsize=(7,5)
)


sns.countplot(
    x=df["cluster"]
)


plt.title(
    "Number of Records per Cluster"
)


plt.xlabel(
    "Cluster"
)


plt.ylabel(
    "Count"
)


plt.show()



# ----------------------------------------------------------
# 15. PCA Visualisation
# ----------------------------------------------------------

pca = PCA(
    n_components=2
)


X_pca = pca.fit_transform(
    X.toarray()
)



pca_df = pd.DataFrame(
    X_pca,
    columns=[
        "PC1",
        "PC2"
    ]
)


pca_df["cluster"] = df["cluster"]



# ----------------------------------------------------------
# 16. Plot Clusters
# ----------------------------------------------------------

plt.figure(
    figsize=(10,6)
)


sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="cluster",
    s=80
)


plt.title(
    "Globe Contours K-Means NLP Clusters"
)


plt.show()



# ----------------------------------------------------------
# 17. Display Cluster Examples
# ----------------------------------------------------------

for c in sorted(
    df["cluster"].unique()
):

    print(
        "\n======================"
    )

    print(
        "CLUSTER:",
        c
    )

    display(
        df[
            df["cluster"] == c
        ].head(5)
    )



# ----------------------------------------------------------
# 18. Save Results
# ----------------------------------------------------------

output_file = (
    "globe_contours_kmeans_results.csv"
)


df.to_csv(
    output_file,
    index=False
)


print(
    "Saved:",
    output_file
)



# ----------------------------------------------------------
# END
# ----------------------------------------------------------

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Libraries loaded successfully
Dataset Loaded
Rows: 145
Columns: 36
<class 'pandas.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 36 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   lat-1   145 non-null    float64
 1   lon-1   145 non-null    float64
 2   lat-2   145 non-null    float64
 3   lon-2   145 non-null    float64
 4   lat-3   145 non-null    float64
 5   lon-3   145 non-null    float64
 6   lat-4   145 non-null    float64
 7   lon-4   145 non-null    float64
 8   lat-5   145 non-null    float64
 9   lon-5   145 non-null    float64
 10  lat-6   145 non-null    float64
 11  lon-6   145 non-null    float64
 12  lat-7   145 non-null    float64
 13  lon-7   145 non-null    float64
 14  lat-8   145 non-null    float64
 15  lon-8   145 non-null    float64
 16  lat-9   145 non-null    float64
 17  lon-9   145 non-null    float64
 18  lat-10  145 non-null    float64
 19  lon-10  145 non-null    float64
 20  lat-11  145 non-null

ValueError: empty vocabulary; perhaps the documents only contain stop words